<a href="https://colab.research.google.com/github/bernardlawes/Colab-Roboflow/blob/main/Roboflow_Ship_Cargo_Sequential.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Inference SDK

In [ ]:
!pip install inference-sdk

# Select Input

In [ ]:
IMAGE_URL = "https://dam.krohne.com/t_ar43_cr_c/e_trim:0/w_auto/q_auto/dpr_auto/f_auto/d_im-other:image-not-available.png/im-contract-photography/orange-black-loaded-container-ship-harbour.jpg"

# Run Inference

In [ ]:
from inference_sdk import InferenceHTTPClient

# Step 1: Initialize Roboflow API Client
client = InferenceHTTPClient(
    api_url="https://detect.roboflow.com",
    api_key=userdata.get('ROBOFLOW_API_KEY')
)


# Step 2: Send Image for Inference
result = client.run_workflow(
    workspace_name="robo-hello-world",
    workflow_id="ship-detect",
    images={
        "image": IMAGE_URL
    },
    use_cache=True # cache workflow definition for 15 minutes
)

# View JSON Result (Pretty Format)

In [ ]:
import json
# Step 3: Display Full JSON (For Debugging)
print(json.dumps(result, indent=4))  # Pretty-print JSON

# Process / Parse JSON Result

In [ ]:
# Step 3: Access First Item (Since it's a list)
data = result[0]

# Step 4: Extract Overall Image Info
image_info = data["predictions"]["image"]
image_width = image_info["width"]
image_height = image_info["height"]

# Step 5: Print Image Size
print(f"Image Size: {image_width}x{image_height}")

print("\n")


# Print Bounding Boxes Info of Ship(s)

In [ ]:
# Step 6: Extract Predictions (List of Detected Objects)
predictions = data["predictions"]["predictions"]

# Step 7: Loop Through Detected Objects and Print Information
for obj in predictions:
    class_name = obj["class"]
    confidence = obj["confidence"] * 100  # Convert to percentage
    x, y = obj["x"], obj["y"]  # Center coordinates
    width, height = obj["width"], obj["height"]  # Bounding box size

    print(f"Detected {class_name}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"Location: x={x}, y={y}")
    print(f"Bounding Box: Width={width}, Height={height}")
    print(f"Detection ID: {obj['detection_id']}")

    print("\n")


# Read in the image from URL

In [ ]:
import cv2
import numpy as np
import requests
from google.colab.patches import cv2_imshow

image_np = np.asarray(bytearray(requests.get(IMAGE_URL).content), dtype=np.uint8)
image = cv2.imdecode(image_np, cv2.IMREAD_COLOR)

# Create a copy of the image that I will use to draw on
image_canvas = image.copy()

# Boundary Box Visualization for Ships

In [ ]:
# Step 4: Draw bounding boxes around detected objects
for obj in predictions:
    x, y, width, height = int(obj["x"]), int(obj["y"]), int(obj["width"]), int(obj["height"])
    class_name = obj["class"]
    confidence = obj["confidence"] * 100  # Convert to percentage

    # Define bounding box color (green) and thickness
    color = (0, 255, 0)
    thickness = 2

    # Draw rectangle around detected object
    top_left = (x - width // 2, y - height // 2)
    bottom_right = (x + width // 2, y + height // 2)
    cv2.rectangle(image_canvas, top_left, bottom_right, color, thickness)

    # Put class name and confidence above bounding box
    label = f"{class_name}: {confidence:.2f}%"
    cv2.putText(image_canvas, label, (top_left[0], top_left[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

# Step 5: Display the result
cv2_imshow(image_canvas)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Crop Visualization of Detected Ship(s)

In [ ]:
# Calculate top-left and bottom-right coordinates
x1 = max(0, x - width // 2)
y1 = max(0, y - height // 2)
x2 = min(image.shape[1], x + width // 2)
y2 = min(image.shape[0], y + height // 2)

# Crop the image
cropped_image = image[y1:y2, x1:x2]

# 🔹 Step 5: Display the result
cv2_imshow(cropped_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Run Inference on the Cropped Image



In [ ]:
from inference_sdk import InferenceHTTPClient

# Step 1: Initialize Roboflow API Client
client = InferenceHTTPClient(
    api_url="https://detect.roboflow.com",
    api_key="puSSAw97Y4qS5df3Xgzc"
)


# Step 2: Send Image for Inference
result = client.run_workflow(
    workspace_name="robo-hello-world",
    workflow_id="cargo-detect",
    images={
        "image": cropped_image
    },
    use_cache=True # cache workflow definition for 15 minutes
)

# Display JSON Response (Optional)

In [ ]:
# Step 3: Display Full JSON (For Debugging)
print(json.dumps(result, indent=4))  # Pretty-print JSON

# Process JSON Result

In [ ]:
# Step 3: Access First Item (Since it's a list)
data = result[0]

# Step 4: Extract General Image Information
image_info = data["model_predictions"]["image"]
image_width = image_info["width"]
image_height = image_info["height"]

# Step 5: Print Image Size
print(f"Image Dimensions: {image_width} x {image_height}")
print("\n")

# Print Bounding Box of the Containers

In [ ]:
# 🔹 Step 6: Extract Predictions (List of Detected Objects)
predictions = data["model_predictions"]["predictions"]

# 🔹 Step 7: Loop Through Detected Objects and Print Information
for obj in predictions:
    class_name = obj["class"]
    confidence = obj["confidence"] * 100  # Convert to percentage
    x, y = obj["x"], obj["y"]  # Center coordinates
    width, height = obj["width"], obj["height"]  # Bounding box size

    print(f"Detected {class_name}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"Location: x={x}, y={y}")
    print(f"Bounding Box: Width={width}, Height={height}")
    print(f"Detection ID: {obj['detection_id']}")

    print("\n")

# Boundary Box Visualization for Cargo / Containers

In [ ]:
# 🔹 Step 4: Draw bounding boxes around detected objects
for obj in predictions:
    x, y, width, height = int(obj["x"]), int(obj["y"]), int(obj["width"]), int(obj["height"])
    class_name = obj["class"]
    confidence = obj["confidence"] * 100  # Convert to percentage

    # Define bounding box color (green) and thickness
    color = (0, 255, 0)
    thickness = 2

    # Draw rectangle around detected object
    top_left = (x - width // 2, y - height // 2)
    bottom_right = (x + width // 2, y + height // 2)
    cv2.rectangle(cropped_image, top_left, bottom_right, color, thickness)

    # Put class name and confidence above bounding box
    label = f"{class_name}: {confidence:.2f}%"
    cv2.putText(cropped_image, label, (top_left[0], top_left[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

# 🔹 Step 5: Display the result
cv2_imshow(image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# STOP EXECUTION HERE

In [ ]:
raise SystemExit("Stopping execution here.")

# Load Google Colab Libs

In [ ]:
!pip install --upgrade google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client
from google.colab import drive
from google.colab import auth
auth.authenticate_user()

# Save Cropped Image(s) in Google Colab Folder

In [ ]:
# Define the filename and save it to Colab
image_path = "/content/saved_image.jpg"
cv2.imwrite(image_path, cropped_image)  # Save image to Colab storage
print(f"Image saved at: {image_path}")

# Dynamically Create Storage Folder in Google Drive

In [ ]:
# Initialize the Drive API client

from googleapiclient.discovery import build
drive_service = build("drive", "v3")

def get_or_create_folder(folder_name):
    """Checks if a folder exists by name, creates it if it doesn't, and returns the folder ID."""
    query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    folders = results.get("files", [])

    # If folder exists, return the ID
    if folders:
        print(f"Folder '{folder_name}' already exists.")
        return folders[0]["id"]

    # If folder doesn't exist, create it
    file_metadata = {
        "name": folder_name,
        "mimeType": "application/vnd.google-apps.folder"
    }
    folder = drive_service.files().create(body=file_metadata, fields="id").execute()
    print(f"Folder '{folder_name}' created.")
    return folder["id"]

# Specify the folder name
folder_name = "MyDynamicFolder"  # Change this to the desired folder name
folder_id = get_or_create_folder(folder_name)

print(f"Folder ID: {folder_id}")

# Upload Saved Image File into Google Drive

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import auth

# Authenticate Google Drive API
auth.authenticate_user()

# Initialize Drive API client
drive_service = build("drive", "v3")

# Set metadata for upload with the folder ID
file_metadata = {
    "name": "saved_image.jpg",
    "mimeType": "image/jpeg",
    "parents": [folder_id]  # Upload to the specified folder
}

# Upload file to Google Drive
media = MediaFileUpload(image_path, mimetype="image/jpeg")
uploaded_file = drive_service.files().create(
    body=file_metadata,
    media_body=media,
    fields="id"
).execute()

# Get uploaded file ID
file_id = uploaded_file.get("id")
print(f"Uploaded File ID: {file_id}")

# Generate a Public URL

In [ ]:
# Generate a shareable link
image_url = f"https://drive.google.com/uc?export=view&id={file_id}"
print(f"Public Image URL: {image_url}")

In [ ]:
# Use if I want to save it to google drive
#drive.mount('/content/drive')
#save_path = "/content/drive/My Drive/Roboflow_Images/cropped001.jpg"

In [ ]:
# Save the image in Colab's working directory
#save_path = "/content/cropped/image001.jpg"

In [ ]:

#cv2.imwrite(save_path, image)
#print(f"File saved at: {save_path}")